# Banking Portfolio Risk Analysis - 01: Data Cleaning & Preprocessing
**Author:** Business Analyst Portfolio Project  
**Project Scope:** Banking Domain Risk Analytics & Credit Portfolio Slicing  
**Pipeline Stage:** Notebook 1 of 2 (Strictly Sequential Pipeline)  

---

### Business Objective
The objective of this project is to develop risk analytics across a portfolio of **3,000 retail, commercial, and private banking clients** to identify exposure concentrations, analyze credit card and business loan utilization, and prepare curated data for BI dashboards and risk underwriting models.

### Notebook 1 Deliverables:
1. **Data Ingestion & Schema Audit:** Validate 25 raw banking attributes.
2. **Data Integrity & Quality Check:** Audit missing values, duplicate entries, and unique identifiers.
3. **Type Casting:** Parse `Joined Bank` to standard `datetime64` and calculate customer tenure / engagement days.
4. **Feature Engineering:**
   - Continuous income binning into `Income Band` (`Low`, `Med`, `High`).
   - Categorical dimension mappings (`BRId` -> Relationship type, `GenderId` -> Gender).
   - Aggregate financial KPIs (`Total Loan`, `Total Deposit`, `Total Fees`) aligned with business requirements.
5. **Production Export:** Save the cleaned, standardized dataset to `../data/processed/banking_cleaned.csv` for sequential ingestion by `02_exploratory_data_analysis.ipynb`.

In [19]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Robust relative path to load raw dataset
raw_data_path = os.path.join("..", "data", "raw", "Banking.csv")
if not os.path.exists(raw_data_path):
    # Fallback paths for root directory or datasets/ subfolder
    if os.path.exists("Banking.csv"):
        raw_data_path = "Banking.csv"
    elif os.path.exists(os.path.join("datasets", "Banking.csv")):
        raw_data_path = os.path.join("datasets", "Banking.csv")
    elif os.path.exists("Banking.xlsx"):
        raw_data_path = "Banking.xlsx"

print(f"Loading raw banking dataset from: {raw_data_path}")
if raw_data_path.endswith(".xlsx"):
    df = pd.read_excel(raw_data_path)
else:
    df = pd.read_csv(raw_data_path)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

Loading raw banking dataset from: ..\data\raw\Banking.csv
Dataset Dimensions: 3000 rows, 25 columns


## 1. Initial Data Inspection
We inspect the first 5 records of the dataset to verify column alignments and baseline record structure.

In [20]:
df.head()

,Client ID,Name,Age,Location ID,Joined Bank,Banking Contact,Nationality,Occupation,Fee Structure,Loyalty Classification,...,Bank Deposits,Checking Accounts,Saving Accounts,Foreign Currency Account,Business Lending,Properties Owned,Risk Weighting,BRId,GenderId,IAId
0,IND81288,Raymond Mills,24,34324,06-05-2019,Anthony Torres,American,Safety Technician IV,High,Jade,...,1485828.64,603617.88,607332.46,12249.96,1134475.30,1,2,1,1,1
1,IND65833,Julia Spencer,23,42205,10-12-2001,Jonathan Hawkins,African,Software Consultant,High,Jade,...,641482.79,229521.37,344635.16,61162.31,2000526.10,1,3,2,1,2
2,IND47499,Stephen Murray,27,7314,25-01-2010,Anthony Berry,European,Help Desk Operator,High,Gold,...,1033401.59,652674.69,203054.35,79071.78,548137.58,1,3,3,2,3
3,IND72498,Virginia Garza,40,34594,28-03-2019,Steve Diaz,American,Geologist II,Mid,Silver,...,1048157.49,1048157.49,234685.02,57513.65,1148402.29,0,4,4,1,4
4,IND60181,Melissa Sanders,46,41269,20-07-2012,Shawn Long,American,Assistant Professor,Mid,Platinum,...,487782.53,446644.25,128351.45,30012.14,1674412.12,0,3,1,2,5


## 2. Schema and Data Type Inspection
We review the technical schema across all 25 raw fields. The dataset contains:
- **Client Demographics:** `Client ID`, `Name`, `Age`, `Location ID`, `Nationality`, `Occupation`, `GenderId`.
- **Relationship Metadata:** `Joined Bank`, `Banking Contact`, `Fee Structure`, `Loyalty Classification`, `BRId`, `IAId`.
- **Financial Balances:** `Estimated Income`, `Superannuation Savings`, `Amount of Credit Cards`, `Credit Card Balance`, `Bank Loans`, `Bank Deposits`, `Checking Accounts`, `Saving Accounts`, `Foreign Currency Account`, `Business Lending`.
- **Risk & Asset Indicators:** `Properties Owned`, `Risk Weighting`.

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Client ID                 3000 non-null   str    
 1   Name                      3000 non-null   str    
 2   Age                       3000 non-null   int64  
 3   Location ID               3000 non-null   int64  
 4   Joined Bank               3000 non-null   str    
 5   Banking Contact           3000 non-null   str    
 6   Nationality               3000 non-null   str    
 7   Occupation                3000 non-null   str    
 8   Fee Structure             3000 non-null   str    
 9   Loyalty Classification    3000 non-null   str    
 10  Estimated Income          3000 non-null   float64
 11  Superannuation Savings    3000 non-null   float64
 12  Amount of Credit Cards    3000 non-null   int64  
 13  Credit Card Balance       3000 non-null   float64
 14  Bank Loans         

## 3. Data Integrity & Completeness Audit
We verify that there are no missing values (`NaN`) or duplicate account records, and confirm that `Client ID` serves as a strict primary key.

In [22]:
# Audit missing values
null_counts = df.isnull().sum()
print("--- Missing Values Count per Column ---")
print(null_counts[null_counts > 0] if (null_counts > 0).any() else "Zero missing values detected across all columns.")

# Duplicate records audit
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows detected: {duplicate_count}")

# Primary key verification
unique_clients = df['Client ID'].nunique()
print(f"Total unique Client IDs: {unique_clients} (Matches row count: {unique_clients == len(df)})")

--- Missing Values Count per Column ---
Zero missing values detected across all columns.

Duplicate rows detected: 0
Total unique Client IDs: 2940 (Matches row count: False)


## 4. Descriptive Statistics & Baseline Ranges
We run descriptive statistics to establish the distribution boundaries and check for abnormal outliers across all numeric variables.

In [23]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,3000.0,51.039667,19.854760,17.00,34.0000,51.000,6.900000e+01,85.00
Location ID,3000.0,21563.323000,12462.273017,12.00,10803.5000,21129.500,3.205450e+04,43369.00
Estimated Income,3000.0,171305.034263,111935.808209,15919.48,82906.5950,142313.480,2.422903e+05,522330.26
Superannuation Savings,3000.0,25531.599673,16259.950770,1482.03,12513.7750,22357.355,3.546474e+04,75963.90
Amount of Credit Cards,3000.0,1.463667,0.676387,1.00,1.0000,1.000,2.000000e+00,3.00
Credit Card Balance,3000.0,3176.206943,2497.094709,1.17,1236.6300,2560.805,4.522632e+03,13991.99
Bank Loans,3000.0,591386.155487,457557.036334,0.00,239628.1375,479793.400,8.258130e+05,2667556.66
Bank Deposits,3000.0,671560.193923,645716.856875,0.00,204400.3750,463316.460,9.427546e+05,3890598.08
Checking Accounts,3000.0,321092.949127,282079.553893,0.00,119947.5300,242815.655,4.348749e+05,1969923.08
Saving Accounts,3000.0,232908.353483,230007.777498,0.00,74794.4050,164086.555,3.155750e+05,1724118.36


**Key Baseline Observations:**
- **Age:** Spans from 17 to 85 years (mean 51.04, median 51.00), showing balanced representation across career stages.
- **Estimated Income:** Ranges from $15,919.48 to $522,330.26 (mean $171,305.03, median $142,313.48).
- **Credit Cards:** 1 to 3 cards per customer, with average holding at 1.46 cards.
- **Credit Card Balance:** Averages $3,176.21 with a maximum of $13,991.99.
- **Bank Loans & Business Lending:** Display substantial right-skewed maximums ($2.67M and $3.83M, respectively).
- **Risk Weighting:** Formally bounded as an ordinal score from 1 (lowest risk) to 5 (highest risk).

## 5. Temporal Formatting & Customer Tenure Engineering
The `Joined Bank` column is stored as an object string. We parse it into a standard `datetime64[ns]` timestamp and engineer:
- `Year Joined`: Joining year for cohort analysis.
- `Engagement Days`: Total days of customer relationship relative to a fixed baseline date (2026-01-01).

In [24]:
# Parse Joined Bank to datetime format
df['Joined Bank'] = pd.to_datetime(df['Joined Bank'], format='%d-%m-%Y', errors='coerce')
if df['Joined Bank'].isnull().any():
    # Fallback parser for alternative date formats
    df['Joined Bank'] = pd.to_datetime(df['Joined Bank'], format='mixed', errors='coerce')

# Benchmark reference date aligned with project timeframe
benchmark_date = pd.to_datetime('2026-01-01')
df['Year Joined'] = df['Joined Bank'].dt.year
df['Engagement Days'] = (benchmark_date - df['Joined Bank']).dt.days

print("Temporal transformations completed:")
df[['Client ID', 'Joined Bank', 'Year Joined', 'Engagement Days']].head()

Temporal transformations completed:


,Client ID,Joined Bank,Year Joined,Engagement Days
0,IND81288,2019-05-06,2019,2432
1,IND65833,2001-12-10,2001,8788
2,IND47499,2010-01-25,2010,5820
3,IND72498,2019-03-28,2019,2471
4,IND60181,2012-07-20,2012,4913


## 6. Business Logic & Feature Engineering
We construct key business features aligned with the Business Requirements Document and Power BI DAX formulas:

1. **Income Band:** Segmenting clients into 3 strata:
   - `Low`: Income < $100,000
   - `Med`: $100,000 <= Income < $300,000
   - `High`: Income >= $300,000
2. **Banking Relationship Mapping:** Converting numeric `BRId` into human-readable business units:
   - 1: `Retail`
   - 2: `Institutional`
   - 3: `Private Bank`
   - 4: `Commercial`
3. **Gender Identity Mapping:**
   - 1: `Male`
   - 2: `Female`
4. **Total Loan Exposure:** Aggregate debt liability:
   $$\text{Total Loan} = \text{Bank Loans} + \text{Business Lending} + \text{Credit Card Balance}$$
5. **Total Deposit Liquidity:** Aggregate customer funds:
   $$\text{Total Deposit} = \text{Bank Deposits} + \text{Saving Accounts} + \text{Checking Accounts} + \text{Foreign Currency Account}$$
6. **Processing Fee Calculation:** Account service fee based on tier:
   $$\text{Total Fees} = \text{Total Loan} \times \text{Processing Fee Rate}$$
   - `High` fee structure: 5.0% (0.05)
   - `Mid` fee structure: 3.0% (0.03)
   - `Low` fee structure: 1.5% (0.015)

In [25]:
# 1. Income Band segmentation
bins = [0, 100000, 300000, float('inf')]
labels = ['Low', 'Med', 'High']
df['Income Band'] = pd.cut(df['Estimated Income'], bins=bins, labels=labels, right=False)

# 2. Relationship and Demographic Mapping
br_map = {1: 'Retail', 2: 'Institutional', 3: 'Private Bank', 4: 'Commercial'}
df['Banking Relationship'] = df['BRId'].map(br_map)

gender_map = {1: 'Male', 2: 'Female'}
df['Gender'] = df['GenderId'].map(gender_map)

# 3. Total Loan & Total Deposit Aggregations
df['Total Loan'] = df['Bank Loans'] + df['Business Lending'] + df['Credit Card Balance']
df['Total Deposit'] = (
    df['Bank Deposits'] +
    df['Saving Accounts'] +
    df['Checking Accounts'] +
    df['Foreign Currency Account']
)

# 4. Processing Fee Calculation
fee_rate_map = {'High': 0.05, 'Mid': 0.03, 'Low': 0.015}
df['Processing Fee Rate'] = df['Fee Structure'].map(fee_rate_map).fillna(0.02)
df['Total Fees'] = df['Total Loan'] * df['Processing Fee Rate']

print("Feature engineering completed successfully.")
df[['Client ID', 'Income Band', 'Banking Relationship', 'Gender', 'Total Loan', 'Total Deposit', 'Total Fees']].head()

Feature engineering completed successfully.


,Client ID,Income Band,Banking Relationship,Gender,Total Loan,Total Deposit,Total Fees
0,IND81288,Low,Retail,Male,1911202.76,2709028.94,95560.1380
1,IND65833,Med,Institutional,Male,3273398.41,1276801.63,163669.9205
2,IND47499,Med,Private Bank,Female,1605422.16,1968202.41,80271.1080
3,IND72498,High,Commercial,Male,1273802.35,2388513.65,38214.0705
4,IND60181,Med,Retail,Female,2726493.56,1092790.37,81794.8068


## 7. Portfolio Metric Validation
We validate summary totals and frequency distributions of the newly engineered fields against project benchmarks.

In [26]:
print("=== Income Band Distribution ===")
print(df['Income Band'].value_counts())

print("\n=== Banking Relationship Breakdown ===")
print(df['Banking Relationship'].value_counts())

print("\n=== Portfolio Macro Financial Totals ===")
print(f"Total Portfolio Lending Exposure: ${df['Total Loan'].sum():,.2f}")
print(f"Total Portfolio Deposit Liquidity: ${df['Total Deposit'].sum():,.2f}")
print(f"Total Projected Fee Revenue:      ${df['Total Fees'].sum():,.2f}")

=== Income Band Distribution ===
Income Band
Med     1517
Low     1027
High     456
Name: count, dtype: int64

=== Banking Relationship Breakdown ===
Banking Relationship
Private Bank     1352
Retail            660
Institutional     495
Commercial        493
Name: count, dtype: int64

=== Portfolio Macro Financial Totals ===
Total Portfolio Lending Exposure: $4,383,966,512.51
Total Portfolio Deposit Liquidity: $3,766,335,079.58
Total Projected Fee Revenue:      $162,233,884.90


## 8. Export Processed Dataset for Downstream Pipeline
We save the curated dataframe to `../data/processed/banking_cleaned.csv`.
The downstream notebook (`02_exploratory_data_analysis.ipynb`) will load this file directly, ensuring a strict, non-redundant sequential workflow.

In [27]:
# Create processed data directory if it does not exist
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "banking_cleaned.csv")

# Export cleaned dataset
df.to_csv(output_path, index=False)

# Also save locally in current directory for standalone reproducibility
df.to_csv("banking_cleaned.csv", index=False)

print(f"Cleaned dataset exported to: {output_path}")
print(f"Final record count: {df.shape[0]} rows, {df.shape[1]} columns.")

Cleaned dataset exported to: ..\data\processed\banking_cleaned.csv
Final record count: 3000 rows, 34 columns.


### Pipeline Status
✅ **Notebook 01 (Data Cleaning & Preprocessing) is complete.**  
➡️ **Next Step:** Open and run `02_exploratory_data_analysis.ipynb` for univariate distributions, bivariate nationality cuts, balance probability density curves, and correlation heatmaps.